# Flafi Trainer (Colab)
Trains flafi on Colab with the current repo layout (`setup.py` + `llm/`). Click **Runtime → Run all**.

Repo is CPU-only (`torch.set_default_device("cpu")`), so Colab GPU won't speed it up — this just gives you a faster CPU and no local wait.

In [ ]:
# @title 0. Setup
!pip install torch numpy -q
import os, sys
# If you opened this notebook from the repo, files are already beside it.
# Otherwise clone it (edit URL to yours):
if not os.path.exists("setup.py"):
    !git clone https://github.com/Shimba-crypto/shimba-llm.git zanji
    %cd zanji
print("cwd:", os.getcwd())
!ls *.py llm/

In [ ]:
# @title 1. Build clean flafi dataset (no long-form poison)
# --include_old "" drops flafi-anonomuse.txt (800-char transformer/RSA answers
# a char model with block 128 can never learn). Pure short turns only.
!python3 gen_flafi.py --out flafi-dataset.txt --chars 200000 --seed 7 --include_old ""
!wc -c flafi-dataset.txt && head -n 12 flafi-dataset.txt

In [ ]:
# @title 2. Train (~1M params, block 128, GPU when available)
# 2000 iters on a Colab GPU is minutes; on CPU ~40-60 min. Lower MAX_ITERS to 500 for a smoke test.
MAX_ITERS = 2000  # @param {type:"integer"}
!python3 setup.py train --data flafi-dataset.txt --out flafi-colab.pth --thinking --device auto \n  --block_size 128 --n_embd 128 --n_layer 4 --n_head 4 \n  --batch_size 8 --grad_accum 4 --lr 3e-4 \n  --max_iters $MAX_ITERS --eval_interval 200 --eval_iters 40

In [ ]:
# @title 3. Smoke test (greedy = truth)
# Greedy (temp 0) must be coherent English. If this soups, sampling tweaks won't save it.
!python3 setup.py generate --model flafi-colab.pth --prompt "User: hi\n" --max_tokens 60 --temperature 0.0
!python3 setup.py generate --model flafi-colab.pth --prompt "User: what is 2 plus 2\n" --max_tokens 80 --temperature 0.0

In [ ]:
# @title 4. Chat + benchmark
!python3 benchmark.py --model flafi-colab.pth 2>&1 | tail -n 30

In [ ]:
# @title 5. Download model
from google.colab import files
files.download("flafi-colab.pth")
files.download("flafi-colab_tokenizer.json")